# LeetCode 674 — Longest Continuous Increasing Subsequence bằng Python

**Link đề:** https://leetcode.com/problems/longest-continuous-increasing-subsequence/
**Độ khó:** Easy · **Chủ đề:** Array

```python
class Solution:
    def findLengthOfLCIS(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Phân biệt **subsequence** (dãy con, có thể nhảy cóc) và **subarray** (đoạn con **liên tiếp**).
2. Viết cách brute force thử mọi điểm bắt đầu và thấy chi phí `O(n²)`.
3. Nắm kỹ thuật **đếm chuỗi chạy** (run length): một biến `cur` tăng khi còn tăng, **reset về 1** khi gãy.
4. Nhìn cùng ý tưởng dưới 2 góc: **mảng DP** `dp[i]` và **cửa sổ/neo** (anchor) `start`.
5. Cẩn thận với "**tăng nghiêm ngặt**" (`<` chứ không phải `<=`).

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho mảng số nguyên `nums` (chưa sắp xếp). Trả về **độ dài** của **đoạn con liên tiếp tăng nghiêm ngặt** dài nhất.

Đoạn `nums[l..r]` hợp lệ nếu `nums[i] < nums[i+1]` với mọi `l <= i < r`.

**Ví dụ 1:** `nums = [1,3,5,4,7]` → `3` (đoạn `[1,3,5]`).
`[1,3,5,7]` tăng nhưng **không liên tiếp** (bị số 4 chen giữa) → không tính.

**Ví dụ 2:** `nums = [2,2,2,2,2]` → `1` (bằng nhau không phải tăng nghiêm ngặt; một phần tử luôn là đoạn tăng dài 1).

**Constraints:** `1 <= len(nums) <= 10^4`, `-10^9 <= nums[i] <= 10^9`.

Vì `len(nums) >= 1` nên đáp án luôn `>= 1`.

In [1]:
nums = [1, 3, 5, 4, 7]
for i in range(len(nums) - 1):
    a, b_ = nums[i], nums[i + 1]
    print(f"cặp ({a}, {b_}): {'tăng' if a < b_ else 'GÃY'}")

cặp (1, 3): tăng
cặp (3, 5): tăng
cặp (5, 4): GÃY
cặp (4, 7): tăng


## 2. Nền tảng Python cần biết

### 2.1. Subarray vs subsequence
- **Subarray** (đoạn con): lấy một khúc **liền nhau**, ví dụ `nums[1:4]`.
- **Subsequence** (dãy con): giữ thứ tự nhưng **được bỏ** phần tử ở giữa.

Bài này hỏi **subarray** (tên đề dùng chữ "continuous subsequence" — chính là subarray).

In [2]:
nums = [1, 3, 5, 4, 7]
print("subarray nums[0:3]:", nums[0:3])
print("subsequence chọn chỉ số 0,1,2,4:", [nums[i] for i in (0, 1, 2, 4)], "-> KHÔNG liên tiếp")

subarray nums[0:3]: [1, 3, 5]
subsequence chọn chỉ số 0,1,2,4: [1, 3, 5, 7] -> KHÔNG liên tiếp


### 2.2. Duyệt cặp kề nhau: `range(1, n)` và `zip(nums, nums[1:])`
So sánh `nums[i-1]` với `nums[i]` cho `i` từ 1 đến `n-1`. Hoặc ghép cặp bằng `zip`.

In [3]:
nums = [1, 3, 5, 4, 7]
for i in range(1, len(nums)):
    print(i, nums[i - 1], "<", nums[i], "?", nums[i - 1] < nums[i])
print(list(zip(nums, nums[1:])))

1 1 < 3 ? True
2 3 < 5 ? True
3 5 < 4 ? False
4 4 < 7 ? True
[(1, 3), (3, 5), (5, 4), (4, 7)]


### 2.3. `<` khác `<=`
"Tăng nghiêm ngặt" (strictly increasing) dùng `<`. Với `[2, 2]`: `2 < 2` là `False` → gãy.

In [4]:
print(2 < 2, 2 <= 2)
print(-10**9 < 10**9)

False True
True


### 2.4. `max()` để giữ kỷ lục
Mẫu "giữ kỷ lục": `best = max(best, cur)` sau mỗi lần `cur` thay đổi.

In [5]:
best = 0
for cur in [1, 2, 3, 1, 2]:
    best = max(best, cur)
    print("cur =", cur, "-> best =", best)

cur = 1 -> best = 1
cur = 2 -> best = 2
cur = 3 -> best = 3
cur = 1 -> best = 3
cur = 2 -> best = 3


## 3. Cách trực giác — Brute Force

Với mỗi vị trí bắt đầu `l`, đi sang phải chừng nào còn tăng; ghi lại độ dài. Lấy max.

```text
với l từ 0 đến n-1:
    r = l
    trong khi r + 1 < n và nums[r] < nums[r+1]: r += 1
    best = max(best, r - l + 1)
```
Tệ nhất (mảng tăng hoàn toàn) tốn `n + (n-1) + ... + 1 ≈ n²/2` bước → `O(n²)`.

In [6]:
def lcis_bruteforce(nums: list[int]) -> int:
    n = len(nums)
    best = 0
    for l in range(n):
        r = l
        while r + 1 < n and nums[r] < nums[r + 1]:
            r += 1
        best = max(best, r - l + 1)
    return best

print(lcis_bruteforce([1, 3, 5, 4, 7]))   # 3
print(lcis_bruteforce([2, 2, 2, 2, 2]))   # 1
print(lcis_bruteforce([1, 2, 3, 4]))      # 4

3
1
4


In [7]:
def count_steps_bruteforce(nums):
    n, steps = len(nums), 0
    for l in range(n):
        r = l
        while r + 1 < n and nums[r] < nums[r + 1]:
            r += 1
            steps += 1
    return steps

for n in [10, 100, 1000]:
    print(f"mảng tăng dài {n}: brute force tốn {count_steps_bruteforce(list(range(n)))} bước")

mảng tăng dài 10: brute force tốn 45 bước
mảng tăng dài 100: brute force tốn 4950 bước
mảng tăng dài 1000: brute force tốn 499500 bước


## 4. Nút thắt và insight

**Nút thắt:** khi bắt đầu từ `l = 0` ta đã đi qua đoạn `[1, 3, 5]`. Bắt đầu lại từ `l = 1`, `l = 2` chỉ đi lại **một phần** của đoạn đó — chắc chắn **ngắn hơn**. Lãng phí!

**Insight:** mảng được chia thành các **đoạn tăng tối đa** (run) bởi các điểm "gãy" (`nums[i-1] >= nums[i]`). Đáp án = độ dài run dài nhất.

Chỉ cần **một biến** `cur` = độ dài đoạn tăng **kết thúc tại `i`**:
- nếu `nums[i-1] < nums[i]` → `cur += 1` (nối dài run);
- ngược lại → `cur = 1` (bắt đầu run mới từ chính `nums[i]`).

In [8]:
nums = [1, 3, 5, 4, 7]
runs, start = [], 0
for i in range(1, len(nums) + 1):
    if i == len(nums) or nums[i - 1] >= nums[i]:
        runs.append(nums[start:i])
        start = i
print("các run tăng tối đa:", runs)
print("độ dài:", [len(r) for r in runs], "-> max =", max(len(r) for r in runs))

các run tăng tối đa: [[1, 3, 5], [4, 7]]
độ dài: [3, 2] -> max = 3


## 5. Lời giải chính — đếm chuỗi chạy (giảng từng đoạn)

**Đoạn 1:** `best = cur = 1` (một phần tử luôn là đoạn tăng dài 1; `n >= 1`).

**Đoạn 2:** duyệt `i` từ 1 đến `n-1`:
- `nums[i-1] < nums[i]` → `cur += 1`
- ngược lại → `cur = 1`

**Đoạn 3:** `best = max(best, cur)` sau mỗi bước.

**Đoạn 4:** trả `best`.

In [9]:
def lcis_run(nums: list[int]) -> int:
    best = cur = 1                         # đoạn 1
    for i in range(1, len(nums)):          # đoạn 2
        if nums[i - 1] < nums[i]:
            cur += 1
        else:
            cur = 1
        best = max(best, cur)              # đoạn 3
    return best                            # đoạn 4

for t in ([1, 3, 5, 4, 7], [2, 2, 2, 2, 2], [5], [1, 2, 3, 4], [4, 3, 2, 1]):
    print(t, "->", lcis_run(t))

[1, 3, 5, 4, 7] -> 3
[2, 2, 2, 2, 2] -> 1
[5] -> 1
[1, 2, 3, 4] -> 4
[4, 3, 2, 1] -> 1


## 6. TRACE từng bước

`nums = [1, 3, 5, 4, 7, 8, 9, 2]`

| `i` | `nums[i-1]` | `nums[i]` | tăng? | `cur` | `best` |
|---:|---:|---:|:---:|---:|---:|
| 0 | – | 1 | – | 1 | 1 |
| 1 | 1 | 3 | ✔ | 2 | 2 |
| 2 | 3 | 5 | ✔ | 3 | 3 |
| 3 | 5 | 4 | ✘ | 1 | 3 |
| 4 | 4 | 7 | ✔ | 2 | 3 |
| 5 | 7 | 8 | ✔ | 3 | 3 |
| 6 | 8 | 9 | ✔ | 4 | **4** |
| 7 | 9 | 2 | ✘ | 1 | 4 |

Kết quả `4` (đoạn `[4, 7, 8, 9]`).

In [10]:
def trace_lcis(nums: list[int]) -> int:
    best = cur = 1
    print(f"i=0: nums[0]={nums[0]} -> cur=1, best=1")
    for i in range(1, len(nums)):
        inc = nums[i - 1] < nums[i]
        cur = cur + 1 if inc else 1
        best = max(best, cur)
        mark = "tăng -> cur+1" if inc else "GÃY -> reset cur=1"
        print(f"i={i}: {nums[i-1]:>2} vs {nums[i]:>2} {mark:<20} cur={cur}, best={best}")
    return best

trace_lcis([1, 3, 5, 4, 7, 8, 9, 2])

i=0: nums[0]=1 -> cur=1, best=1
i=1:  1 vs  3 tăng -> cur+1        cur=2, best=2
i=2:  3 vs  5 tăng -> cur+1        cur=3, best=3
i=3:  5 vs  4 GÃY -> reset cur=1   cur=1, best=3
i=4:  4 vs  7 tăng -> cur+1        cur=2, best=3
i=5:  7 vs  8 tăng -> cur+1        cur=3, best=3
i=6:  8 vs  9 tăng -> cur+1        cur=4, best=4
i=7:  9 vs  2 GÃY -> reset cur=1   cur=1, best=4


4

### 6.1. Trace với phần tử bằng nhau
`[1, 2, 2, 3]`: cặp `(2, 2)` không tăng → reset. Đáp án `2`, không phải `4` hay `3`.

In [11]:
trace_lcis([1, 2, 2, 3])

i=0: nums[0]=1 -> cur=1, best=1
i=1:  1 vs  2 tăng -> cur+1        cur=2, best=2
i=2:  2 vs  2 GÃY -> reset cur=1   cur=1, best=2
i=3:  2 vs  3 tăng -> cur+1        cur=2, best=2


2

## 7. Code nộp LeetCode

### 7.1. Cách 1 — Đếm chuỗi chạy (khuyên dùng)

In [12]:
class Solution:
    def findLengthOfLCIS(self, nums: list[int]) -> int:
        best = cur = 1
        for i in range(1, len(nums)):
            if nums[i - 1] < nums[i]:
                cur += 1
                if cur > best:
                    best = cur
            else:
                cur = 1
        return best

print(Solution().findLengthOfLCIS([1, 3, 5, 4, 7]))
print(Solution().findLengthOfLCIS([2, 2, 2, 2, 2]))

3
1


### 7.2. Cách 2 — Neo (anchor) / cửa sổ trượt

Giữ `anchor` = chỉ số bắt đầu của run hiện tại. Gặp điểm gãy tại `i` (`nums[i-1] >= nums[i]`) → `anchor = i`. Độ dài run kết thúc tại `i` là `i - anchor + 1`.

Đây là cách lời giải chính thức của LeetCode trình bày — tương đương `cur = i - anchor + 1`.

In [13]:
class SolutionAnchor:
    def findLengthOfLCIS(self, nums: list[int]) -> int:
        best = anchor = 0
        for i in range(len(nums)):
            if i > 0 and nums[i - 1] >= nums[i]:
                anchor = i
            best = max(best, i - anchor + 1)
        return best

print(SolutionAnchor().findLengthOfLCIS([1, 3, 5, 4, 7]))
print(SolutionAnchor().findLengthOfLCIS([2, 2, 2, 2, 2]))

3
1


### 7.3. Cách 3 — Mảng DP

`dp[i]` = độ dài đoạn tăng dài nhất **kết thúc tại `i`**.
- `dp[0] = 1`
- `dp[i] = dp[i-1] + 1` nếu `nums[i-1] < nums[i]`, ngược lại `1`.
- Đáp án `max(dp)`.

Biến `cur` ở cách 1 chính là `dp[i]` — ta chỉ cần giá trị trước nên **nén** mảng thành 1 biến (`O(n)` → `O(1)` bộ nhớ).

In [14]:
def lcis_dp(nums: list[int]) -> int:
    n = len(nums)
    dp = [1] * n
    for i in range(1, n):
        if nums[i - 1] < nums[i]:
            dp[i] = dp[i - 1] + 1
    print("dp =", dp)
    return max(dp)

print(lcis_dp([1, 3, 5, 4, 7, 8, 9, 2]))

dp = [1, 2, 3, 1, 2, 3, 4, 1]
4


In [15]:
class SolutionDP:
    def findLengthOfLCIS(self, nums: list[int]) -> int:
        dp = [1] * len(nums)
        for i in range(1, len(nums)):
            if nums[i - 1] < nums[i]:
                dp[i] = dp[i - 1] + 1
        return max(dp)

print(SolutionDP().findLengthOfLCIS([1, 3, 5, 4, 7]))

3


### 7.4. Cách 4 — `itertools.groupby` (Pythonic)
Đánh dấu mỗi cặp kề nhau là tăng (`True`) hay không. Đoạn tăng dài `L` ứng với `L-1` giá trị `True` **liên tiếp**. `groupby` gom các giá trị giống nhau đứng liền.

In [16]:
from itertools import groupby

flags = [a < b_ for a, b_ in zip([1, 3, 5, 4, 7], [3, 5, 4, 7])]
print("flags:", flags)
print("nhóm:", [(k, len(list(g))) for k, g in groupby(flags)])

class SolutionGroupby:
    def findLengthOfLCIS(self, nums: list[int]) -> int:
        flags = (a < b_ for a, b_ in zip(nums, nums[1:]))
        longest_true = max((len(list(g)) for k, g in groupby(flags) if k), default=0)
        return longest_true + 1

print(SolutionGroupby().findLengthOfLCIS([1, 3, 5, 4, 7]), SolutionGroupby().findLengthOfLCIS([2, 2, 2]), SolutionGroupby().findLengthOfLCIS([9]))

flags: [True, True, False, True]
nhóm: [(True, 2), (False, 1), (True, 1)]
3 1 1


## 8. Vì sao đúng?

**Bất biến:** sau khi xử lý chỉ số `i`,
1. `cur` = độ dài đoạn tăng nghiêm ngặt **dài nhất kết thúc tại `i`**;
2. `best` = độ dài đoạn tăng dài nhất **kết thúc tại một chỉ số `≤ i`**.

- **Khởi tạo** `i = 0`: đoạn duy nhất kết thúc tại 0 là `[nums[0]]` → `cur = best = 1`.
- **Duy trì:** nếu `nums[i-1] < nums[i]`, đoạn dài nhất kết thúc tại `i` = đoạn dài nhất kết thúc tại `i-1` nối thêm `nums[i]` → `cur + 1`. Nếu không, mọi đoạn dài ≥ 2 kết thúc tại `i` đều chứa cặp `(i-1, i)` không tăng → chỉ còn đoạn `[nums[i]]` → `cur = 1`.
- **Kết thúc:** mọi đoạn tăng đều kết thúc ở một chỉ số nào đó → `best` là đáp án.

## 9. Độ phức tạp và so sánh

Gọi `n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---:|---:|---|
| Brute force mọi `l` | `O(n²)` | `O(1)` | Đi lại các run nhiều lần |
| Đếm chuỗi chạy `cur` | `O(n)` | `O(1)` | **Khuyên dùng** |
| Neo `anchor` | `O(n)` | `O(1)` | Dễ mở rộng để trả về **vị trí** |
| Mảng DP | `O(n)` | `O(n)` | Dạy tư duy DP |
| `groupby` | `O(n)` | `O(n)` (`nums[1:]`) | Gọn, khó đọc với người mới |

## 10. Bộ test

1. Ví dụ đề.
2. Một phần tử; hai phần tử bằng nhau / tăng / giảm.
3. Tăng toàn bộ; giảm toàn bộ.
4. Run dài nhất ở **cuối** mảng (bẫy quên cập nhật `best` sau vòng lặp).
5. Số âm lớn, phần tử bằng nhau xen giữa.
6. Ngẫu nhiên: đối chiếu mọi cách với brute force.

In [17]:
import random

cases = [
    ([1, 3, 5, 4, 7], 3), ([2, 2, 2, 2, 2], 1), ([5], 1),
    ([1, 1], 1), ([1, 2], 2), ([2, 1], 1),
    ([1, 2, 3, 4, 5], 5), ([5, 4, 3, 2, 1], 1),
    ([3, 1, 2, 3, 4], 4),                      # run dài nhất ở cuối
    ([-10**9, 0, 10**9], 3), ([1, 2, 2, 3, 4], 3),
]
solvers = [Solution().findLengthOfLCIS, SolutionAnchor().findLengthOfLCIS, SolutionDP().findLengthOfLCIS,
           SolutionGroupby().findLengthOfLCIS, lcis_bruteforce, lcis_run]
for nums, expected in cases:
    for f in solvers:
        assert f(nums) == expected, (f, nums)
    print(f"{str(nums):<28} -> {expected} PASS")

rng = random.Random(674)
for _ in range(500):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(1, 15))]
    exp = lcis_bruteforce(nums)
    assert all(f(nums) == exp for f in solvers)
print("500 mảng ngẫu nhiên: mọi cách khớp brute force PASS")

[1, 3, 5, 4, 7]              -> 3 PASS
[2, 2, 2, 2, 2]              -> 1 PASS
[5]                          -> 1 PASS
[1, 1]                       -> 1 PASS
[1, 2]                       -> 2 PASS
[2, 1]                       -> 1 PASS
[1, 2, 3, 4, 5]              -> 5 PASS
[5, 4, 3, 2, 1]              -> 1 PASS
[3, 1, 2, 3, 4]              -> 4 PASS
[-1000000000, 0, 1000000000] -> 3 PASS
[1, 2, 2, 3, 4]              -> 3 PASS
500 mảng ngẫu nhiên: mọi cách khớp brute force PASS


## 11. Lỗi người mới hay gặp

### Lỗi 1: Chỉ cập nhật `best` khi gặp điểm gãy
Nếu run dài nhất nằm ở **cuối** mảng, vòng lặp kết thúc mà chưa có điểm gãy → `best` không được cập nhật.

In [18]:
def wrong_update_on_break(nums):
    best = cur = 1
    for i in range(1, len(nums)):
        if nums[i - 1] < nums[i]:
            cur += 1
        else:
            best = max(best, cur)     # SAI: chỉ cập nhật khi gãy
            cur = 1
    return best

print("sai:", wrong_update_on_break([3, 1, 2, 3, 4]), "| đúng:", Solution().findLengthOfLCIS([3, 1, 2, 3, 4]))

sai: 1 | đúng: 4


### Lỗi 2: Dùng `<=` (cho phép bằng nhau)
`[2, 2, 2]` sẽ cho `3` thay vì `1`.

### Lỗi 3: Reset `cur = 0` thay vì `1`
Phần tử tại điểm gãy **bắt đầu** run mới, nên run mới dài 1 chứ không phải 0.

In [19]:
def wrong_le(nums):
    best = cur = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i - 1] <= nums[i] else 1     # SAI
        best = max(best, cur)
    return best

def wrong_reset_zero(nums):
    best = cur = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i - 1] < nums[i] else 0      # SAI
        best = max(best, cur)
    return best

print("<= sai:", wrong_le([2, 2, 2]), "| đúng:", Solution().findLengthOfLCIS([2, 2, 2]))
print("reset 0 sai:", wrong_reset_zero([5, 1, 2, 3]), "| đúng:", Solution().findLengthOfLCIS([5, 1, 2, 3]))

<= sai: 3 | đúng: 1
reset 0 sai: 2 | đúng: 3


### Lỗi 4: Nhầm với Longest Increasing **Subsequence** (LeetCode 300)
Bài 300 cho phép nhảy cóc → cần DP `O(n²)` hoặc patience sorting `O(n log n)`. Áp dụng lời giải bài 300 vào đây cho `[1,3,5,4,7]` ra `4` — sai với bài 674.

### Lỗi 5: So sánh `nums[i]` với `nums[i+1]` nhưng duyệt `range(len(nums))`
→ `IndexError` ở phần tử cuối. Dùng `range(1, n)` và so `nums[i-1]`, hoặc `range(n-1)` và so `nums[i+1]`.

In [20]:
def lis_length(nums):          # LeetCode 300 — dãy con (không cần liên tiếp), DP O(n^2)
    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)

print("LIS (bài 300):", lis_length([1, 3, 5, 4, 7]), "| LCIS (bài 674):", Solution().findLengthOfLCIS([1, 3, 5, 4, 7]))
try:
    nums = [1, 2, 3]
    for i in range(len(nums)):
        _ = nums[i] < nums[i + 1]
except IndexError as e:
    print("IndexError:", e)

LIS (bài 300): 4 | LCIS (bài 674): 3
IndexError: list index out of range


## 12. Mẹo nhận dạng pattern

- **"Đoạn liên tiếp dài nhất thỏa điều kiện giữa hai phần tử kề nhau"** → đếm chuỗi chạy: `cur = cur + 1 if ok(i) else 1`.
- Đây là dạng DP đơn giản nhất: `dp[i]` chỉ phụ thuộc `dp[i-1]` → nén thành 1 biến.
- Các bài cùng khuôn: 485 Max Consecutive Ones, 1446 Consecutive Characters, 978 Longest Turbulent Subarray, 2414 Longest Alphabetical Continuous Substring.
- Phân biệt: "subsequence" thật sự (nhảy cóc) → bài 300 LIS, khó hơn.

## 13. Biến thể

### 13.1. Trả về chính đoạn tăng dài nhất (đầu tiên nếu hòa)
Dùng cách neo: nhớ `anchor` khi `best` được cập nhật.

In [21]:
def lcis_segment(nums):
    best, best_l, anchor = 0, 0, 0
    for i in range(len(nums)):
        if i > 0 and nums[i - 1] >= nums[i]:
            anchor = i
        if i - anchor + 1 > best:
            best, best_l = i - anchor + 1, anchor
    return nums[best_l:best_l + best]

print(lcis_segment([1, 3, 5, 4, 7, 8, 9, 2]))   # [4, 7, 8, 9]

[4, 7, 8, 9]


### 13.2. Max Consecutive Ones (LeetCode 485)
Cùng khuôn "đếm chuỗi chạy", điều kiện là `x == 1`.

In [22]:
def max_consecutive_ones(nums):
    best = cur = 0
    for x in nums:
        cur = cur + 1 if x == 1 else 0
        best = max(best, cur)
    return best

print(max_consecutive_ones([1, 1, 0, 1, 1, 1]))   # 3

3


## 14. Bài tập (tự làm trước, đáp án ở cuối)

**Bài 1 — Tính tay:** lập bảng `cur`, `best` cho `nums = [5, 6, 3, 4, 5, 1, 2]`.

**Bài 2 — Đoán output:** `lcis_dp([3, 3, 4, 5, 5, 6])` in mảng `dp` nào?

**Bài 3 — TODO:** đoạn liên tiếp **giảm nghiêm ngặt** dài nhất.

In [23]:
def exercise_longest_decreasing(nums: list[int]) -> int:
    # TODO: giống lời giải chính, đổi điều kiện so sánh
    pass

# assert exercise_longest_decreasing([5, 4, 3, 4, 2]) == 3
# assert exercise_longest_decreasing([1, 1]) == 1

**Bài 4 — TODO:** đoạn liên tiếp **không giảm** (cho phép bằng nhau) dài nhất.

In [24]:
def exercise_longest_non_decreasing(nums: list[int]) -> int:
    # TODO
    pass

# assert exercise_longest_non_decreasing([1, 2, 2, 3, 1]) == 4
# assert exercise_longest_non_decreasing([3, 2, 1]) == 1

**Bài 5 — TODO:** trả về **số lượng** đoạn tăng tối đa (run) có độ dài bằng đáp án.

In [25]:
def exercise_count_longest_runs(nums: list[int]) -> int:
    # TODO: tách mảng thành các run tăng tối đa, lưu độ dài từng run, đếm số run có độ dài lớn nhất
    pass

# assert exercise_count_longest_runs([1, 2, 0, 3, 4, 2]) == 1      # chỉ có [0,3,4] dài 3
# assert exercise_count_longest_runs([1, 2, 1, 2, 1, 2]) == 3

**Bài 6 — Mini challenge (LeetCode 978 — Longest Turbulent Subarray):** đoạn liên tiếp dài nhất mà dấu so sánh **đổi chiều** liên tục (`a < b > c < d ...` hoặc ngược lại).

In [26]:
def exercise_turbulent(arr: list[int]) -> int:
    # TODO: giữ hai biến up (kết thúc bằng tăng) và down (kết thúc bằng giảm)
    pass

# assert exercise_turbulent([9, 4, 2, 10, 7, 8, 8, 1, 9]) == 5
# assert exercise_turbulent([4, 8, 12, 16]) == 2
# assert exercise_turbulent([100]) == 1

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Bắt đầu `cur = best = 1` | Gãy tại 6→3 và 5→1 | best cuối = 3 |
| 2 | `dp[0] = 1` | 3→3 không tăng | `[1, 1, 2, 3, 1, 2]` |
| 3 | Đổi `<` thành `>` | `nums[i-1] > nums[i]` | phần còn lại giữ nguyên |
| 4 | Đổi `<` thành `<=` | Lần này bằng nhau vẫn nối | `nums[i-1] <= nums[i]` |
| 5 | Liệt kê các run tăng tối đa | Ghi độ dài mỗi run khi gặp điểm gãy (và run cuối) | `lengths.count(max(lengths))` |
| 6 | Mỗi bước biết "cặp cuối tăng hay giảm" | `up = down + 1` nếu tăng; `down = up + 1` nếu giảm | bằng nhau → cả hai về 1 |

## 16. Đáp án tham khảo

**Bài 1:**

| i | cặp | cur | best |
|---:|---|---:|---:|
| 0 | – | 1 | 1 |
| 1 | 5<6 | 2 | 2 |
| 2 | 6>3 | 1 | 2 |
| 3 | 3<4 | 2 | 2 |
| 4 | 4<5 | 3 | 3 |
| 5 | 5>1 | 1 | 3 |
| 6 | 1<2 | 2 | 3 |

**Bài 2:** `dp = [1, 1, 2, 3, 1, 2]`, kết quả `3`.

**Bài 5:** với `[1, 2, 0, 3, 4, 2]` các run là `[1,2]`, `[0,3,4]`, `[2]` → dài nhất 3, có **1** run. Với `[1,2,1,2,1,2]` có 3 run dài 2. Cách an toàn nhất: tách mảng thành danh sách độ dài các run rồi đếm (xem code dưới).

In [27]:
trace_lcis([5, 6, 3, 4, 5, 1, 2])
print("Bài 2:", lcis_dp([3, 3, 4, 5, 5, 6]))

i=0: nums[0]=5 -> cur=1, best=1
i=1:  5 vs  6 tăng -> cur+1        cur=2, best=2
i=2:  6 vs  3 GÃY -> reset cur=1   cur=1, best=2
i=3:  3 vs  4 tăng -> cur+1        cur=2, best=2
i=4:  4 vs  5 tăng -> cur+1        cur=3, best=3
i=5:  5 vs  1 GÃY -> reset cur=1   cur=1, best=3
i=6:  1 vs  2 tăng -> cur+1        cur=2, best=3
dp = [1, 1, 2, 3, 1, 2]
Bài 2: 3


In [28]:
def answer_longest_decreasing(nums):
    best = cur = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i - 1] > nums[i] else 1
        best = max(best, cur)
    return best


def answer_longest_non_decreasing(nums):
    best = cur = 1
    for i in range(1, len(nums)):
        cur = cur + 1 if nums[i - 1] <= nums[i] else 1
        best = max(best, cur)
    return best


def answer_count_longest_runs(nums):
    # Tách mảng thành các run tăng tối đa, rồi đếm run có độ dài lớn nhất.
    lengths, cur = [], 1
    for i in range(1, len(nums)):
        if nums[i - 1] < nums[i]:
            cur += 1
        else:
            lengths.append(cur)
            cur = 1
    lengths.append(cur)
    best = max(lengths)
    return lengths.count(best)


def answer_turbulent(arr):
    best = up = down = 1
    for i in range(1, len(arr)):
        if arr[i - 1] < arr[i]:
            up, down = down + 1, 1
        elif arr[i - 1] > arr[i]:
            up, down = 1, up + 1
        else:
            up = down = 1
        best = max(best, up, down)
    return best


assert answer_longest_decreasing([5, 4, 3, 4, 2]) == 3
assert answer_longest_decreasing([1, 1]) == 1
assert answer_longest_non_decreasing([1, 2, 2, 3, 1]) == 4
assert answer_longest_non_decreasing([3, 2, 1]) == 1
assert answer_count_longest_runs([1, 2, 0, 3, 4, 2]) == 1      # chỉ có [0,3,4]
assert answer_count_longest_runs([1, 2, 1, 2, 1, 2]) == 3
assert answer_turbulent([9, 4, 2, 10, 7, 8, 8, 1, 9]) == 5
assert answer_turbulent([4, 8, 12, 16]) == 2
assert answer_turbulent([100]) == 1
for _ in range(300):
    nums = [rng.randint(-3, 3) for _ in range(rng.randint(1, 12))]
    assert answer_longest_decreasing(nums) == Solution().findLengthOfLCIS([-x for x in nums])
print("Đáp án tham khảo: tất cả assert PASS")

Đáp án tham khảo: tất cả assert PASS


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findLengthOfLCIS(self, nums: list[int]) -> int:
        best = cur = 1
        for i in range(1, len(nums)):
            cur = cur + 1 if nums[i - 1] < nums[i] else 1
            best = max(best, cur)
        return best
```

**Câu thần chú:** *"Còn tăng thì cộng, gãy thì về 1, kỷ lục cập nhật mỗi bước."*

**Checklist:**
- [ ] Dùng `<` (tăng **nghiêm ngặt**).
- [ ] Reset `cur = 1`, không phải `0`.
- [ ] Cập nhật `best` **mỗi bước** (hoặc thêm lần cuối sau vòng lặp).
- [ ] Không nhầm với bài 300 (dãy con không liên tiếp).
- [ ] Time `O(n)`, Space `O(1)`.